# Storyboard-to-Video Pipeline
**Built on WAN 2.1 (Alibaba) — Course Project Demo**

**Full pipeline:**
1. Scene prompts define each shot with cinematic detail
2. CogVideoX-2b (WAN-compatible API) generates a video clip per scene
3. MoviePy stitches all clips into one final video

> **Before running:** Runtime -> Change runtime type -> **T4 GPU**

> **Note:** CogVideoX-2b is used as the free-GPU proxy. WAN 2.1 uses the same HuggingFace Diffusers API — swapping models is a one-line change.

## Step 1 — Check GPU & Install Dependencies

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — go to Runtime > Change runtime type > T4 GPU')

Fri May  1 14:14:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate
!pip install -q imageio imageio-ffmpeg moviepy
!apt-get install -qq ffmpeg
print('All dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 32.5 MB/s eta 0:00:00
All dependencies installed


## Step 2 — Scene Prompts (Storyboard)

Each scene is a structured dict with a title and a detailed cinematic prompt.
The prompt encodes subject, camera motion, lighting, depth of field, and visual style.

**Future scope:** Replace this hardcoded list with an LLM call (Gemini / GPT-4o)
that takes a one-line story idea from the user and auto-generates these scene prompts.

In [ ]:
SCENES = [
    {
        "title": "Scene 1 — The Field",
        "prompt": (
            "A field of bright red poppies and yellow wildflowers swaying gently in a warm breeze. "
            "Camera slowly pushes forward at ground level through the flowers. "
            "Clear blue sky, soft golden sunlight, green stems, vibrant colors. "
            "Medium wide shot, slow dolly forward, cheerful spring atmosphere, cinematic 4K."
        ),
    },
    {
        "title": "Scene 2 — The Sunflower",
        "prompt": (
            "A row of tall bright yellow sunflowers slowly swaying in a light breeze against a clear blue sky. "
            "Camera gently pans left to right across the flowers. "
            "Warm afternoon sunlight, petals glowing gold, green leaves. "
            "Medium shot, slow pan, peaceful and warm atmosphere, cinematic."
        ),
    },
    {
        "title": "Scene 3 — The Butterfly",
        "prompt": (
            "A colorful orange butterfly perched on a pink flower, slowly opening and closing its wings. "
            "Soft green garden background, warm sunlight, gentle breeze moves the flower slightly. "
            "Medium close-up shot, shallow depth of field, soft bokeh background, slow motion, cinematic."
        ),
    },
]

NEGATIVE_PROMPT = (
    "dark tones, overexposed, static, blurred details, subtitles, text, watermark, "
    "pond5, getty, shutterstock, stock footage, logo, worst quality, low quality, "
    "jpeg artifacts, ugly, deformed, blurry, gloomy, rain, tractor, vehicle, person"
)

# Video settings
NUM_FRAMES = 17   # ~2s at 8fps — keep low for T4 (15GB VRAM)
FPS        = 8
HEIGHT     = 480
WIDTH      = 720

print(f'Loaded {len(SCENES)} scenes')
for i, s in enumerate(SCENES):
    print(f'  [{i+1}] {s["title"]}')
    print(f'      {s["prompt"][:90]}...')
print(f'Settings: {NUM_FRAMES} frames @ {FPS}fps, {WIDTH}x{HEIGHT}')

Loaded 3 scenes
  [1] Scene 1 — The Field
      A field of bright red poppies and yellow wildflowers swaying gently in a warm breeze. Came...
  [2] Scene 2 — The Sunflower
      A row of tall bright yellow sunflowers slowly swaying in a light breeze against a clear bl...
  [3] Scene 3 — The Butterfly
      A colorful orange butterfly perched on a pink flower, slowly opening and closing its wings...
Settings: 17 frames @ 8fps, 720x480


## Step 3 — Load Model

Using **CogVideoX-2b** as the generation backbone — same HuggingFace Diffusers API as WAN 2.1.

Memory optimizations required for free T4 (15GB VRAM):
- `torch.float16` — half-precision weights, halves VRAM vs float32
- `enable_sequential_cpu_offload()` — moves layers CPU<->GPU one at a time
- `vae.enable_tiling()` — splits video into spatial tiles during VAE decode
- `vae.enable_slicing()` — processes frames in chunks to prevent OOM

In [ ]:
import torch
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

MODEL_ID = "THUDM/CogVideoX-2b"
# To use WAN 2.1 on an A100, swap to: "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

print(f'Loading {MODEL_ID}...')
pipe = CogVideoXPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()
print('Model ready')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading THUDM/CogVideoX-2b...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Model ready


## Step 4 — Generate One Clip Per Scene

Key generation parameters:
- `num_inference_steps=25` — denoising iterations (quality vs speed tradeoff)
- `guidance_scale=6.0` — CFG weight controlling prompt adherence vs diversity
- `num_frames=17` — temporal depth (~2s at 8fps)

In [ ]:
import time

generated_clips = []

for i, scene in enumerate(SCENES):
    scene_num = i + 1
    output_path = f"scene_{scene_num}.mp4"

    print(f'\n{scene["title"]} ({scene_num}/{len(SCENES)})')
    print(f'  Prompt: {scene["prompt"][:90]}...')

    t0 = time.time()
    output = pipe(
        prompt=scene["prompt"],
        negative_prompt=NEGATIVE_PROMPT,
        num_videos_per_prompt=1,
        num_inference_steps=25,
        num_frames=NUM_FRAMES,
        guidance_scale=6.0,
        height=HEIGHT,
        width=WIDTH,
    ).frames[0]

    export_to_video(output, output_path, fps=FPS)
    elapsed = time.time() - t0
    print(f'  Saved {output_path} ({elapsed:.0f}s)')
    generated_clips.append(output_path)

print(f'\nAll {len(generated_clips)} clips generated!')


Scene 1 — The Field (1/3)
  Prompt: A field of bright red poppies and yellow wildflowers swaying gently in a warm breeze. Came...


  0%|          | 0/25 [00:00<?, ?it/s]

  Saved scene_1.mp4 (435s)

Scene 2 — The Sunflower (2/3)
  Prompt: A row of tall bright yellow sunflowers slowly swaying in a light breeze against a clear bl...


  0%|          | 0/25 [00:00<?, ?it/s]

  Saved scene_2.mp4 (432s)

Scene 3 — The Butterfly (3/3)
  Prompt: A colorful orange butterfly perched on a pink flower, slowly opening and closing its wings...


  0%|          | 0/25 [00:00<?, ?it/s]

  Saved scene_3.mp4 (434s)

All 3 clips generated!


## Step 5 — Stitch into Final Video

MoviePy concatenates the clips on CPU — no GPU needed for this step.
This is what we run live during the demo.

In [ ]:
from moviepy.editor import VideoFileClip, concatenate_videoclips

FINAL_OUTPUT = "final_video.mp4"

print('Stitching clips...')
clips = [VideoFileClip(p) for p in generated_clips]
final = concatenate_videoclips(clips, method="compose")
final.write_videofile(FINAL_OUTPUT, codec="libx264", audio=False, verbose=False, logger=None)
for c in clips:
    c.close()

print(f'Final video saved: {FINAL_OUTPUT}')
print(f'Total duration: {final.duration:.1f}s')

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



Stitching clips...


  warnings.warn("Warning: in file %s, "%(self.filename)+



Final video saved: final_video.mp4
Total duration: 6.4s


## Step 6 — Preview & Download

In [ ]:
from IPython.display import Video, display

print('=== Individual scenes ===')
for path in generated_clips:
    print(f'\n{path}')
    display(Video(path, embed=True, width=640))

=== Individual scenes ===

scene_1.mp4



scene_2.mp4



scene_3.mp4


In [ ]:
print('=== Final stitched video ===')
display(Video(FINAL_OUTPUT, embed=True, width=640))

=== Final stitched video ===


In [ ]:
from google.colab import files
print('Downloading final video...')
files.download(FINAL_OUTPUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Tips & Troubleshooting

**Out of VRAM (OOM)?**
- Reduce `NUM_FRAMES` to 9
- Reduce `WIDTH x HEIGHT` to `480 x 360`
- Both `vae.enable_tiling()` and `vae.enable_slicing()` must be enabled

**Want to use WAN 2.1 instead?**
- Requires an A100 runtime (Colab Pro)
- Change MODEL_ID to `Wan-AI/Wan2.1-T2V-1.3B-Diffusers`
- Change `CogVideoXPipeline` to `WanPipeline`
- Use `torch.bfloat16` instead of `float16`
- Increase `NUM_FRAMES` to 49, `FPS` to 16, `WIDTH` to 832

**Prompt tips for cinematic results:**
- Camera motion: *slow push in, aerial tracking, dolly zoom, handheld*
- Lighting: *golden hour, foggy backlight, neon-lit, overcast*
- Style anchors: *cinematic, 4K, photorealistic, film grain*
- Always end with a style anchor — it strongly influences the output

**Future scope:**
Replace the hardcoded `SCENES` list with an LLM call that accepts a one-line
story idea from the user and auto-generates structured scene prompts with Gemini or GPT-4o.